# Aula ML 04 — Árvores, florestas e a comparação

Previsão do preço de **53.920 diamantes**: quando a relação entre as variáveis deixa de ser uma reta e as categorias se cruzam.

Neste caderno percorremos seis modelos no **mesmo split de teste** (13.480 pedras), acompanhamos o comportamento de um diamante concreto (o **Diamante 157**), diagnosticamos o *overfitting* na prática e medimos a importância real de cada variável por permutação.


## 1. O Dataset: Diamonds

Das 53.940 linhas brutas do Kaggle, 20 pedras contêm `x`, `y` ou `z` iguais a zero (erro de medição dos paquímetros). Removemos essas 20 antes de qualquer split.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pedras_bruto = pd.read_csv("dados/diamonds.csv")
print(f"Linhas brutas: {len(pedras_bruto)}")

# Descarte das 20 pedras com dimensão zero
pedras = pedras_bruto[(pedras_bruto["x"] > 0) & (pedras_bruto["y"] > 0) & (pedras_bruto["z"] > 0)].copy()
print(f"Linhas válidas: {len(pedras)} (removidas {len(pedras_bruto) - len(pedras)} com dimensão zero)")

pedras.head()


## 2. Onde a reta quebra: dispersão Preço × Quilate

A relação entre peso (quilate) e preço tem formato curvo. Além disso, o corte desloca os preços: diamantes do mesmo peso custam muito mais se o corte for *Ideal* do que se for *Fair*.


In [ ]:
amostra = pedras.sample(4000, random_state=0)
fig, ax = plt.subplots(figsize=(7, 3.8))
for corte, cor in zip(
    ["Fair", "Good", "Very Good", "Premium", "Ideal"],
    ["#8a90a0", "#d9820b", "#5b56e0", "#0a9d70", "#14161a"]
):
    ped = amostra[amostra["cut"] == corte]
    ax.scatter(ped["carat"], ped["price"], s=7, c=cor, alpha=0.55, linewidths=0, label=corte)

ax.set_xlabel("Quilates (carat)")
ax.set_ylabel("Preço (US$)")
ax.set_title("Preço contra Quilate (amostra colorida pelo corte)")
ax.legend(frameon=False, fontsize=9, markerscale=2)
plt.show()


## 3. Divisão Treino/Teste e Pipeline de Preparo

- **75% para treino** (40.440 pedras) e **25% para teste** (13.480 pedras) com `random_state=0`.
- O preparo aplica `StandardScaler` nas 6 colunas numéricas e `OneHotEncoder(drop='first')` nas 3 colunas categóricas (`cut`, `color`, `clarity`).


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

alvo = "price"
numericas = ["carat", "depth", "table", "x", "y", "z"]
categoricas = ["cut", "color", "clarity"]

X = pedras[numericas + categoricas]
y = pedras[alvo]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=0
)

print(f"Treino: {len(X_treino)} pedras | Teste: {len(X_teste)} pedras")

def criar_preparo():
    return ColumnTransformer([
        ("num", StandardScaler(), numericas),
        ("cat", OneHotEncoder(drop="first", sparse_output=False), categoricas),
    ])


## 4. Setup do MLflow e Funções de Métrica

Registramos todas as runs no experimento `aula-ml-4-diamonds` para comparar de forma reprodutível no MLflow.


In [ ]:
import mlflow
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.pipeline import Pipeline

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
mlflow.set_tracking_uri("mlruns")
mlflow.set_experiment("aula-ml-4-diamonds")

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def mae(y_true, y_pred):
    return float(mean_absolute_error(y_true, y_pred))

resultados = {}
modelos_treinados = {}

def registrar_modelo(nome, estimador):
    pipe = Pipeline([
        ("preparo", criar_preparo()),
        ("modelo", estimador)
    ])
    pipe.fit(X_treino, y_treino)
    
    prev_teste = pipe.predict(X_teste)
    prev_treino = pipe.predict(X_treino)
    
    metricas = {
        "teste_rmse": rmse(y_teste, prev_teste),
        "teste_mae": mae(y_teste, prev_teste),
        "treino_rmse": rmse(y_treino, prev_treino),
    }
    
    with mlflow.start_run(run_name=nome):
        mlflow.log_params({"modelo": nome})
        mlflow.log_metrics(metricas)
        
    resultados[nome] = metricas
    modelos_treinados[nome] = pipe
    print(f"{nome:20s} -> RMSE Teste: {metricas['teste_rmse']:.1f} | Treino: {metricas['treino_rmse']:.1f}")
    return pipe


### Modelo 1: Regressão Linear (A Reta Baseline)

O modelo de referência herdado da aula 2. Cada feature recebe um coeficiente fixo.


In [ ]:
from sklearn.linear_model import LinearRegression

pipe_reta = registrar_modelo("reta", LinearRegression())


### Modelo 2: Árvore de Decisão Rasa (`max_depth=3`)

Faz no máximo 3 perguntas consecutivas. Fácil de desenhar e visualizar.


In [ ]:
from sklearn.tree import DecisionTreeRegressor, plot_tree

pipe_arvore_rasa = registrar_modelo("arvore-rasa", DecisionTreeRegressor(max_depth=3, random_state=0))

# Desenho da árvore rasa
fig, ax = plt.subplots(figsize=(12, 4.5))
nomes_feat = pipe_arvore_rasa.named_steps["preparo"].get_feature_names_out()
plot_tree(
    pipe_arvore_rasa.named_steps["modelo"],
    feature_names=nomes_feat,
    filled=False,
    impurity=False,
    fontsize=7,
    ax=ax,
    max_depth=3
)
plt.title("Árvore de Decisão Rasa (profundidade 3)")
plt.show()


### Modelo 3: Árvore Funda (Sem limite de profundidade)

Sem limite, ela continua dividindo até que as folhas tenham 1 ou 2 diamantes. Observe a queda brutal do RMSE de treino (overfitting).


In [ ]:
pipe_arvore_funda = registrar_modelo("arvore-funda", DecisionTreeRegressor(random_state=0))


### Modelo 4: Random Forest (Floresta Aleatória)

Conjunto de 80 árvores com *bagging* e subamostragem aleatória de variáveis. A resposta final é a média das 80 árvores.


In [ ]:
from sklearn.ensemble import RandomForestRegressor

pipe_floresta = registrar_modelo(
    "floresta",
    RandomForestRegressor(n_estimators=80, max_depth=16, n_jobs=-1, random_state=0)
)


### Modelo 5: Gradient Boosting

80 árvores treinadas sequencialmente, cada uma prevendo o resíduo (erro) deixado pelo conjunto até aquele ponto.


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

pipe_boosting = registrar_modelo(
    "boosting",
    GradientBoostingRegressor(n_estimators=80, max_depth=3, random_state=0)
)


### Modelo 6: K-Nearest Neighbors (`k=15`)

Prevê a média dos 15 diamantes mais próximos no espaço padronizado de features.


In [ ]:
from sklearn.neighbors import KNeighborsRegressor

pipe_knn = registrar_modelo("knn", KNeighborsRegressor(n_neighbors=15))


## 5. Comparação dos Modelos no Teste

Avaliamos todos os 6 modelos nas mesmas 13.480 pedras de teste.


In [ ]:
tabela = pd.DataFrame(resultados).T
tabela = tabela[["teste_rmse", "teste_mae", "treino_rmse"]]
tabela.columns = ["RMSE Teste (US$)", "MAE Teste (US$)", "RMSE Treino (US$)"]
tabela = tabela.sort_values("RMSE Teste (US$)")
tabela


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)

nomes_modelos = tabela.index
rmse_te = tabela["RMSE Teste (US$)"]
rmse_tr = tabela["RMSE Treino (US$)"]

# Painel 1: RMSE no teste
ax1.barh(nomes_modelos, rmse_te, color=["#0a9d70", "#5b56e0", "#2563eb", "#dc2626", "#8a90a0", "#d9820b"])
for i, v in enumerate(rmse_te):
    ax1.text(v + 15, i, f"{v:.0f}", va="center", fontweight="bold")
ax1.set_title("RMSE no Teste (menor é melhor)")
ax1.set_xlim(0, 1650)
ax1.invert_yaxis()

# Painel 2: Treino vs Teste
y_pos = np.arange(len(nomes_modelos))
h = 0.35
ax2.barh(y_pos - h/2, rmse_tr, height=h, label="Treino", color="#94a3b8")
ax2.barh(y_pos + h/2, rmse_te, height=h, label="Teste", color="#3b82f6")
ax2.set_title("Treino vs Teste (Overfitting)")
ax2.set_xlim(0, 1650)
ax2.legend()

plt.tight_layout()
plt.savefig("img/g-comparacao.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. O Diamante 157 do Teste nos Seis Modelos

Inspecionamos a pedra na **posição 157** do conjunto de teste: um diamante nobre de **2,01 quilates**, corte *Ideal*, cor *G*, pureza *SI2*.
Preço real no mercado: **US$ 18.259**.


In [ ]:
pedra_idx = 157
diamante = X_teste.iloc[[pedra_idx]]
preco_real = y_teste.iloc[pedra_idx]

print("Diamante 157:")
for k, v in diamante.iloc[0].to_dict().items():
    print(f"  {k:10s}: {v}")
print(f"\nPreço Real: US$ {preco_real:,.2f}")


### 6.1 Regressão Linear: Pesos e Contribuições

Como a reta calcula o preço a partir da soma ponderada dos coeficientes padronizados:


In [ ]:
reg = pipe_reta.named_steps["modelo"]
prep = pipe_reta.named_steps["preparo"]
nomes_colunas = prep.get_feature_names_out()
diamante_trans = prep.transform(diamante)[0]

print(f"Intercepto b: {reg.intercept_:.2f}")
contribuicoes = sorted(
    zip(nomes_colunas, reg.coef_, diamante_trans, reg.coef_ * diamante_trans),
    key=lambda x: abs(x[3]),
    reverse=True
)

for nome_c, w, val, contrib in contribuicoes[:6]:
    print(f"  {nome_c:25s} -> val: {val:+.2f} * peso: {w:+.2f} = {contrib:+.2f}")

prev_reta = pipe_reta.predict(diamante)[0]
print(f"\nPrevisão da Reta: US$ {prev_reta:,.2f} (Erro: US$ {prev_reta - preco_real:,.2f})")


### 6.2 Árvore Rasa: O Caminho das Perguntas

As 3 perguntas respondidas pelo Diamante 157 até a folha:


In [ ]:
tree = pipe_arvore_rasa.named_steps["modelo"].tree_
node = 0
while tree.children_left[node] != tree.children_right[node]:
    f_idx = tree.feature[node]
    f_name = nomes_colunas[f_idx]
    limiar = tree.threshold[node]
    val = diamante_trans[f_idx]
    decisao = val <= limiar
    print(f"Nó {node}: {f_name} <= {limiar:.3f}? (valor={val:.3f}) -> {'SIM' if decisao else 'NÃO'}")
    node = tree.children_left[node] if decisao else tree.children_right[node]

folha_val = tree.value[node][0][0]
n_amostras = tree.n_node_samples[node]
print(f"Chegou na folha {node}: {n_amostras} pedras do treino, valor médio US$ {folha_val:,.2f}")


### 6.3 Random Forest: O Voto das 80 Árvores

A média das 80 árvores independentes:


In [ ]:
rf_mod = pipe_floresta.named_steps["modelo"]
rf_trans = pipe_floresta.named_steps["preparo"].transform(diamante)
votos_arvores = [t.predict(rf_trans)[0] for t in rf_mod.estimators_]

print(f"Menor voto: US$ {min(votos_arvores):,.2f}")
print(f"Maior voto: US$ {max(votos_arvores):,.2f}")
print(f"Média das 80 árvores (Floresta): US$ {np.mean(votos_arvores):,.2f}")
print(f"Previsão oficial: US$ {pipe_floresta.predict(diamante)[0]:,.2f}")


### 6.4 KNN: Os 15 Vizinhos Mais Próximos no Treino

Quais pedras reais do treino foram selecionadas pelo KNN como mais parecidas:


In [ ]:
knn_mod = pipe_knn.named_steps["modelo"]
diamante_knn_trans = pipe_knn.named_steps["preparo"].transform(diamante)
dists, indices = knn_mod.kneighbors(diamante_knn_trans)

vizinhos_df = X_treino.iloc[indices[0][:5]].copy()
vizinhos_df["distancia"] = dists[0][:5]
vizinhos_df["preco_real"] = y_treino.iloc[indices[0][:5]].values
vizinhos_df[["carat", "cut", "color", "clarity", "distancia", "preco_real"]]


## 7. Importância de Variáveis por Permutação

Qual coluna realmente manda no preço? Embaralhamos cada coluna no teste da Floresta e medimos quanto o RMSE explode.


In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    pipe_floresta,
    X_teste,
    y_teste,
    n_repeats=3,
    random_state=0,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

ordem = np.argsort(perm.importances_mean)[::-1]
plt.figure(figsize=(7, 3.2))
plt.barh(
    [X.columns[i] for i in ordem][::-1],
    perm.importances_mean[ordem][::-1],
    color="#5b56e0"
)
plt.xlabel("Piora no RMSE de teste ao embaralhar a coluna (US$)")
plt.title("Importância por Permutação no Teste (Floresta)")
plt.show()


## 8. Conclusão e Resumo Prático

1. **Floresta Aleatória (Random Forest)** foi a grande vencedora da aula, com **RMSE de US$ 624** no teste (contra US$ 1.142 da reta).
2. **A árvore funda sem limite** memorizou o treino (RMSE de treino 9 contra teste 821), evidenciando o perigo da variância descontrolada.
3. **Quilates e pureza (clarity)** dominam o preço final do diamante, como provado pela análise de permutação.
4. Para abrir o painel do MLflow e auditar as métricas visualmente:
   ```bash
   MLFLOW_ALLOW_FILE_STORE=true mlflow ui --backend-store-uri mlruns
   ```
